# Setup
For dev, you must have the backend api running on your computer. For prod, please change USER_API_URL to reflect the production url.

In [21]:
import requests
import json
import os
import re
import pprint as pp
from dotenv import load_dotenv
from bson.objectid import ObjectId
from datetime import datetime
from functools import reduce
from pymongo import MongoClient, ReturnDocument, UpdateOne
from pymongo.errors import BulkWriteError

load_dotenv()
custom_request_header = os.getenv("CUSTOM_REQUEST_HEADER")
DATABASE_URL = os.getenv("DATABASE_URL")

OPERATE_ON_LIVE_DATA = True

# Connect to database and check current list of DBs

In [8]:
# Connect to MongoDB
client = MongoClient(DATABASE_URL)
print(client.list_database_names())

['backup_db', 'testdb', 'vrms-populate-projects-test', 'vrms-slack-dev', 'vrms-slack-main', 'vrms-slack-staging', 'vrms-test', 'vrms-test-2', 'vrms-test-3', 'vrms-test-4', 'vrms-test-5', 'vrms-test-6', 'vrms-test-clone-project-sync', 'vrms-test-copy', 'vrms-test-sync', 'vrms-user-migration-test', 'admin', 'local']


# Create a new test database

Define a source and copy for databases


In [22]:
print(f"Operate on live data is: {OPERATE_ON_LIVE_DATA}")

Operate on live data is: True


In [23]:
db_source = client['vrms-test']
db_copy = None

if OPERATE_ON_LIVE_DATA:
    db_copy = client['vrms-test']
else:
    db_copy = client['vrms-populate-projects-test']
print(db_copy)

Database(MongoClient(host=['cluster0-shard-00-01.5buwz.mongodb.net:27017', 'cluster0-shard-00-02.5buwz.mongodb.net:27017', 'cluster0-shard-00-00.5buwz.mongodb.net:27017'], document_class=dict, tz_aware=False, connect=True, retrywrites=True, w='majority', authsource='admin', replicaset='atlas-rl9oiw-shard-0', tls=True), 'vrms-test')


# Drop all collections in test database (ONLY IF NECESSARY!)


In [12]:
if OPERATE_ON_LIVE_DATA == False:
    for collection_name in db_copy.list_collection_names():
        db_copy.drop_collection(collection_name)
        print(f"Dropped collection: {collection_name}")
    print(db_copy.list_collection_names())
else:
    print('Skipping this step')

Dropped collection: users
Dropped collection: projects
[]


# Copy Users and Projects collections from source -> test databases


In [25]:
users_collection = db_source['users']
users = list(users_collection.find())
projects_collection = db_source['projects']
projects = list(projects_collection.find())

users_copy = db_copy['users']
projects_copy = db_copy['projects']

try:
    if OPERATE_ON_LIVE_DATA == False:
        users_copy.insert_many(users, ordered=False) # Copy source db users to test db users
        projects_copy.insert_many(projects, ordered=False) # Copy source db projects to test db projects
        print(db_copy.list_collection_names())
    else:
        print('Skipping database insertions')
except BulkWriteError as bwe:
    print("BulkWriteError details:")
    print(bwe.details)  # This contains info on which documents failed and why


Skipping this step


# Get Users with at least one managedProjects

Retrieve a list of all users with at least one managedProject.


In [26]:
query = {
  "managedProjects": { 
      "$exists": True, 
      "$not": { "$size": 0 } 
  }
}

target_users = list(users_copy.find(query))
pp.pprint(target_users)

[{'__v': 0,
  '_id': ObjectId('6481155fab091f001e30925b'),
  'accessLevel': 'admin',
  'createdDate': datetime.datetime(2023, 6, 7, 23, 40, 15, 196000),
  'currentRole': 'Product Manager',
  'desiredRole': 'Product Manager',
  'email': 'jhaeger30@gmail.com',
  'firstAttended': 'JUN 2023',
  'managedProjects': ['68a3e64ee2653c001fe3ff3b'],
  'name': {'firstName': 'Jack', 'lastName': 'Haeger'},
  'newMember': False,
  'projects': [],
  'skillsToMatch': [],
  'textingOk': False},
 {'__v': 0,
  '_id': ObjectId('66024c13e6a0050028e07948'),
  'accessLevel': 'user',
  'createdDate': datetime.datetime(2024, 3, 26, 4, 16, 19, 45000),
  'currentRole': 'PM',
  'desiredRole': 'PM',
  'email': 'jack.haeger@gmail.com',
  'firstAttended': 'MAR 2024',
  'isActive': True,
  'managedProjects': ['68a3e64ee2653c001fe3ff3b'],
  'name': {'firstName': 'Jack', 'lastName': 'Haeger-PM'},
  'newMember': True,
  'projects': [],
  'skillsToMatch': [],
  'textingOk': False},
 {'__v': 3,
  '_id': ObjectId('670dd397c

# Create an dictionary called `projects_users`

The dict has project IDs as keys and arrays of user IDs as values


In [27]:
projects_users = {}

# Function to filter only projects with valid mongoose IDs
def filter_valid_mongoose_ids(id_list):
    return [x for x in id_list if ObjectId.is_valid(x)]

for user in target_users:
    # Destructure id and managed projects from user
    _id, managed_projects = user['_id'], user['managedProjects']

    # Filter projects
    filtered_projects = filter_valid_mongoose_ids(managed_projects)

    for proj_id in filtered_projects:
        if proj_id in projects_users:
            projects_users[f"{proj_id}"].append(_id)
        else:
            projects_users[f"{proj_id}"] = [_id]

pp.pprint(projects_users)

{'5edeac78ce228b001778facd': [ObjectId('68d1ffc4e2653c001fe400c9')],
 '68a3e64ee2653c001fe3ff3b': [ObjectId('6481155fab091f001e30925b'),
                              ObjectId('66024c13e6a0050028e07948'),
                              ObjectId('670dd397cace6a002abb20ce')],
 '68a3e75ea19d60385b3938f8': [ObjectId('670dd397cace6a002abb20ce')]}


# Update `managedByUsers` field in Projects 

Update all project's `managedByUsers` array using bulk write

In [28]:
operations = []

for proj_id, user_ids in projects_users.items():
    valid_user_ids = [uid for uid in user_ids if ObjectId.is_valid(uid)]    

    proj = projects_copy.find_one({"_id": ObjectId(proj_id)})

    if proj:
        print('Project before update:')
        pp.pprint(proj)
        
        # Compile individual updates in operations 
        operations.append(UpdateOne(
            {"_id": ObjectId(proj_id)}, # Filter
            {"$set": {"managedByUsers": valid_user_ids}}, # Update
        ))
    else:
        print(f"No project with {proj_id} found")

# Execute the bulk write to update operations
result = projects_copy.bulk_write(operations)

print(f"Result: ", result)

Project before update:
{'__v': 0,
 '_id': ObjectId('68a3e64ee2653c001fe3ff3b'),
 'createdDate': datetime.datetime(2025, 8, 19, 2, 49, 50, 843000),
 'description': 'Testing...',
 'githubIdentifier': 'lkjlkj',
 'githubUrl': 'lkjlk',
 'googleDriveUrl': 'https://drive.google.com/drive/folders/1hAq0wyZKOaZLujqOYiaFv5PYgooISger?usp=drive_link',
 'hflaWebsiteUrl': 'lkjlkj',
 'managedByUsers': [],
 'name': 'Jacks Test Project',
 'partners': [],
 'projectStatus': 'Active',
 'recruitingCategories': [],
 'slackUrl': 'lkjlkj'}
Project before update:
{'__v': 0,
 '_id': ObjectId('68a3e75ea19d60385b3938f8'),
 'createdDate': datetime.datetime(2025, 8, 19, 2, 54, 22, 871000),
 'description': 'afk',
 'githubIdentifier': 'afk',
 'githubUrl': 'afk',
 'googleDriveUrl': 'https://drive.google.com/test',
 'hflaWebsiteUrl': 'afk',
 'managedByUsers': [],
 'name': 'VRMS Test Project',
 'partners': [],
 'projectStatus': 'Active',
 'recruitingCategories': [],
 'slackUrl': 'afk'}
Project before update:
{'__v': 0,
 

## Confirm Projects have been updated

In [29]:
for proj_id, user_ids in projects_users.items():
    proj = projects_copy.find_one({"_id": ObjectId(proj_id)})
    if proj:
        print('Project after update:')
        pp.pprint(proj)

Project after update:
{'__v': 0,
 '_id': ObjectId('68a3e64ee2653c001fe3ff3b'),
 'createdDate': datetime.datetime(2025, 8, 19, 2, 49, 50, 843000),
 'description': 'Testing...',
 'githubIdentifier': 'lkjlkj',
 'githubUrl': 'lkjlk',
 'googleDriveUrl': 'https://drive.google.com/drive/folders/1hAq0wyZKOaZLujqOYiaFv5PYgooISger?usp=drive_link',
 'hflaWebsiteUrl': 'lkjlkj',
 'managedByUsers': [ObjectId('6481155fab091f001e30925b'),
                    ObjectId('66024c13e6a0050028e07948'),
                    ObjectId('670dd397cace6a002abb20ce')],
 'name': 'Jacks Test Project',
 'partners': [],
 'projectStatus': 'Active',
 'recruitingCategories': [],
 'slackUrl': 'lkjlkj'}
Project after update:
{'__v': 0,
 '_id': ObjectId('68a3e75ea19d60385b3938f8'),
 'createdDate': datetime.datetime(2025, 8, 19, 2, 54, 22, 871000),
 'description': 'afk',
 'githubIdentifier': 'afk',
 'githubUrl': 'afk',
 'googleDriveUrl': 'https://drive.google.com/test',
 'hflaWebsiteUrl': 'afk',
 'managedByUsers': [ObjectId('67